In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
from google.colab import files

# 1. Please make sure your current runtime utilizes GPU

Recommended GPU: G4 (NVIDIA RTX PRO 6000 Blackwell)

In [ ]:
!nvidia-smi

Sun Jun  7 01:59:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   31C    P0             48W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
!pip install -U kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.8/132.8 kB 600.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.0/230.0 kB 5.1 MB/s eta 0:00:00
  Attempting uninstall: kagglesdk
    Found existing installation: kagglesdk 0.1.23
    Uninstalling kagglesdk-0.1.23:
      Successfully uninstalled kagglesdk-0.1.23
  Attempting uninstall: kaggle
    Found existing installation: kaggle 2.0.2
    Uninstalling kaggle-2.0.2:
      Successfully uninstalled kaggle-2.0.2


#2. Kaggle Setup

## 1. Generate API Token
1. Go to the Kaggle website
2. Log in into your account
3. Create a new Kaggle API **Legacy** token
4. You should now have kaggle.json

## 2. Upload kaggle.json
1. In the Google Colab GUI click on the file icon on the left side of the screen: This is a temporary /content directory, it gets deleted each time you disconnect from runtime
2. Run the next cell and select kaggle.json to upload to this directory
3. In the cell output you should now see kaggle.json


**Example cell output:**

kaggle.json(application/json) - 64 bytes, last modified: 6/4/2026 - 100% done

Saving kaggle.json to kaggle.json

['.config', 'drive', 'kaggle.json', 'sample_data']


In [ ]:
uploaded = files.upload()
print(os.listdir())

Saving kaggle.json to kaggle.json
['.config', 'drive', 'kaggle.json', 'sample_data']


## 3. Login
1. Run the next cell
2. If you can see a list of competitions appear, it means you've logged in successfully

In [ ]:
import shutil

# Set up kaggle credentials
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
shutil.copy('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

# Verify
!kaggle competitions list --search whale

ref                                                                                   deadline             category        reward  teamCount  userHasEntered  
------------------------------------------------------------------------------------  -------------------  ----------  ----------  ---------  --------------  
https://www.kaggle.com/competitions/humpback-whale-identification                     2019-02-28 23:59:00  Featured    25,000 Usd       2120           False  
https://www.kaggle.com/competitions/happy-whale-and-dolphin                           2022-04-18 23:59:00  Research    25,000 Usd       1588            True  
https://www.kaggle.com/competitions/noaa-right-whale-recognition                      2016-01-07 23:59:00  Research    10,000 Usd        364           False  
https://www.kaggle.com/competitions/whale-categorization-playground                   2018-07-09 23:59:00  Playground       Kudos        527           False  
https://www.kaggle.com/competitions/BT5153-202

#3. Create Project Directory in the content/happywhale
1. Create a corresponding folder
2. Clone reference GitHub repository

In [ ]:
PROJECT_DIR = '/content/happywhale'
os.makedirs(PROJECT_DIR, exist_ok=True)
%cd {PROJECT_DIR}

/content/happywhale


In [ ]:
!git clone https://github.com/knshnb/kaggle-happywhale-1st-place.git
%cd kaggle-happywhale-1st-place

Cloning into 'kaggle-happywhale-1st-place'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 57 (delta 10), reused 9 (delta 9), pack-reused 39 (from 1)
Receiving objects: 100% (57/57), 3.90 MiB | 21.80 MiB/s, done.
Resolving deltas: 100% (20/20), done.
/content/happywhale/kaggle-happywhale-1st-place


In [ ]:
%cd {PROJECT_DIR}/kaggle-happywhale-1st-place/input

/content/happywhale/kaggle-happywhale-1st-place/input


#4. Download necessary datasets

In [ ]:
!kaggle competitions download -c happy-whale-and-dolphin

100% 57.7G/57.7G [07:12<00:00, 143MB/s]



In [ ]:
!unzip -q happy-whale-and-dolphin.zip
!rm happy-whale-and-dolphin.zip

In [ ]:
!kaggle datasets download -d jpbremer/fullbodywhaleannotations
!unzip -q fullbodywhaleannotations.zip
!rm fullbodywhaleannotations.zip

Dataset URL: https://www.kaggle.com/datasets/jpbremer/fullbodywhaleannotations
License(s): CC0-1.0
100% 28.3G/28.3G [03:39<00:00, 139MB/s]



#5. Install Required Packages

In [ ]:
!pip install \
    optuna \
    optuna-integration[pytorch_lightning] \
    pytorch-lightning==2.5.0 \
    timm==1.0.3 \
    albumentations==1.4.0 \
    faiss-gpu-cu12 \
    wandb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 857.2 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.4/819.4 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.6/123.6 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.4/103.4 kB 10.0 MB/s eta 0:00:00
  Attempting uninstall: albumentations
    Found existing installation: albumentations 2.0.8
    Uninstalling albumentations-2.0.8:
      Successfully uninstalled albumentations-2.0.8
  Attempting 

In [ ]:
!apt-get install -y libxcb1 libxcb-shm0 libxcb-icccm4 libxcb-image0 libxcb-keysyms1 libxcb-randr0 libxcb-render-util0 libxcb-xinerama0 libxcb-xfixes0 libgl1 libglib2.0-0

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libgl1 is already the newest version (1.4.0-1).
libxcb-randr0 is already the newest version (1.14-3ubuntu3).
libxcb-randr0 set to manually installed.
libxcb-shm0 is already the newest version (1.14-3ubuntu3).
libxcb-shm0 set to manually installed.
libxcb-xfixes0 is already the newest version (1.14-3ubuntu3).
libxcb-xfixes0 set to manually installed.
libxcb1 is already the newest version (1.14-3ubuntu3).
libxcb1 set to manually installed.
libglib2.0-0 is already the newest version (2.72.4-0ubuntu2.9).
The following additional packages will be installed:
  libxcb-util1
The following NEW packages will be installed:
  libxcb-icccm4 libxcb-image0 libxcb-keysyms1 libxcb-render-util0 libxcb-util1
  libxcb-xinerama0
0 upgraded, 6 newly installed, 0 to remove and 53 not upgraded.
Need to get 58.8 kB of archives.
After this operation, 279 kB of additional disk space will be used.
Get:1 http://archive

In [ ]:
%cd ..

/content/happywhale/kaggle-happywhale-1st-place



#6. Upload Necessary Files

All necessary files can be found in the https://github.com/p-oleksii/Group13_Happywhale GitHub repository.

You can either clone the repository to your local machine using the following command:


```
git clone https://github.com/p-oleksii/Group13_Happywhale.git
```



Or you can download them manually using the .zip option on GitHub.


---



After downloading all necessary files, you need to upload them to the current Google Colab session in order to allow proper execution. In order to do that, please follow the instruction below:


##1. Upload configuration files

Configuration files are stored in "config" folder in .yaml format and are responsible for defining model-specific parameters such as resolution, learning rate, etc.

Please upload all configuration files available in our repository.


**Example cell output:**

```
Saving efficientnet_b6_824_continue.yaml to efficientnet_b6_824_continue.yaml
Saving efficientnet_b6_1024_continue.yaml to efficientnet_b6_1024_continue.yaml
Saving efficientnet_b6_1280_continue.yaml to efficientnet_b6_1280_continue.yaml
Saving efficientnet_b6_1356_continue.yaml to efficientnet_b6_1356_continue.yaml
Saving efficientnet_b6_1440_continue.yaml to efficientnet_b6_1440_continue.yaml
Saving efficientnet_b7_768.yaml to efficientnet_b7_768.yaml
Saving efficientnet_b7_1024_ft.yaml to efficientnet_b7_1024_ft.yaml
Saving efficientnet_b7_1024_resume.yaml to efficientnet_b7_1024_resume.yaml
Saving efficientnet_b7_1280_ft.yaml to efficientnet_b7_1280_ft.yaml
Saving efficientnet_b7_1280_resume .yaml to efficientnet_b7_1280_resume .yaml
 config.py                            efficientnet_b6.yaml
 debug.yaml                           efficientnet_b7_1024_ft.yaml
 default.yaml                         efficientnet_b7_1024_resume.yaml
 efficientnet_b5.yaml                 efficientnet_b7_1280_ft.yaml
 efficientnet_b6_1024_continue.yaml  'efficientnet_b7_1280_resume .yaml'
 efficientnet_b6_1280_continue.yaml   efficientnet_b7_768.yaml
 efficientnet_b6_1356_continue.yaml   efficientnet_b7.yaml
 efficientnet_b6_1440_continue.yaml   efficientnet_v2l.yaml
 efficientnet_b6_824_continue.yaml    efficientnet_v2m.yaml
```



In [ ]:
%cd config
uploaded = files.upload()
%ls
%cd ..

/content/happywhale/kaggle-happywhale-1st-place/config


Saving efficientnet_b6_824_continue.yaml to efficientnet_b6_824_continue.yaml
Saving efficientnet_b6_1024_continue.yaml to efficientnet_b6_1024_continue.yaml
Saving efficientnet_b6_1280_continue.yaml to efficientnet_b6_1280_continue.yaml
Saving efficientnet_b6_1356_continue.yaml to efficientnet_b6_1356_continue.yaml
Saving efficientnet_b6_1440_continue.yaml to efficientnet_b6_1440_continue.yaml
Saving efficientnet_b7_768.yaml to efficientnet_b7_768.yaml
Saving efficientnet_b7_1024_ft.yaml to efficientnet_b7_1024_ft.yaml
Saving efficientnet_b7_1024_resume.yaml to efficientnet_b7_1024_resume.yaml
Saving efficientnet_b7_1280_ft.yaml to efficientnet_b7_1280_ft.yaml
Saving efficientnet_b7_1280_resume .yaml to efficientnet_b7_1280_resume .yaml
 config.py                            efficientnet_b6.yaml
 debug.yaml                           efficientnet_b7_1024_ft.yaml
 default.yaml                         efficientnet_b7_1024_resume.yaml
 efficientnet_b5.yaml                 efficientnet_b7_1

##2. Upload source files
This files are crucial for training and define our custom pipeline. Source files are stored in "src" folder in .py format.

**Example cell output:**


```
Saving dataset.py to dataset.py
Saving train_custom.py to train_custom.py
dataset.py  ensemble.py  metric_learning.py  train_custom.py  tune.py  utils.py
```



In [ ]:
%cd src
!rm -rf dataset.py
!rm -rf train.py
uploaded = files.upload()
%ls
%cd ..

/content/happywhale/kaggle-happywhale-1st-place/src


Saving dataset.py to dataset.py
Saving train_custom.py to train_custom.py
dataset.py  ensemble.py  metric_learning.py  train_custom.py  tune.py  utils.py
/content/happywhale/kaggle-happywhale-1st-place


##3. Upload the input files
Additional dataset input files correspond to bboxes for the Backfin dataset. Additional input files are stored inside the input folder in .csv format.

**Example cell output:**



```
Saving test_backfin.csv to test_backfin.csv
Saving test2.csv to test2.csv
Saving train_backfin.csv to train_backfin.csv
Saving train2.csv to train2.csv
fullbody_annotations.csv  pseudo_labels/         test_images/
fullbody_test_charm.csv   README.md              train/
fullbody_test.csv         sample_submission.csv  train2.csv
fullbody_train_charm.csv  species.npy*           train_backfin.csv
fullbody_train.csv        test/                  train.csv
individual_id.npy*        test2.csv              train_images/
meta.json                 test_backfin.csv
```



In [ ]:
%cd input
uploaded = files.upload()
%ls
%cd ..

/content/happywhale/kaggle-happywhale-1st-place/input


Saving test_backfin.csv to test_backfin.csv
Saving test2.csv to test2.csv
Saving train_backfin.csv to train_backfin.csv
Saving train2.csv to train2.csv
fullbody_annotations.csv  pseudo_labels/         test_images/
fullbody_test_charm.csv   README.md              train/
fullbody_test.csv         sample_submission.csv  train2.csv
fullbody_train_charm.csv  species.npy*           train_backfin.csv
fullbody_train.csv        test/                  train.csv
individual_id.npy*        test2.csv              train_images/
meta.json                 test_backfin.csv
/content/happywhale/kaggle-happywhale-1st-place


# 7. Run Training

## 1. Specify the Correct Configuration

Before starting training, ensure that:

* The correct model is selected.
* The correct configuration file (`--config_path`) is specified.

---

## 2. Resume Training from the Last Checkpoint

If you would like to resume training using `--load_snapshot`, the checkpoint file **must be named exactly**:

```text
last.ckpt
```

Any other filename will not be recognized automatically.

---

## 3. Load Weights Only

To load only the model weights, use:

```bash
--load_weights_only
```

This option is intended for continuing training with a **different configuration of the same model architecture**.

### Example

Suppose you want to train an EfficientNet-B6 model in two stages:

1. **1024 × 1024 resolution for 30 epochs**
2. **1280 × 1280 resolution for additional training**

### Stage 1: Initial Training (1024 × 1024, 30 Epochs)

```bash
!python -m src.train_custom \
    --config_path config/efficientnet_b6.yaml \
    --in_base_dir input \
    --exp_name b6 \
    --save_checkpoint
```

### Stage 2: Continue Training (1280 × 1280)

```bash
!python -m src.train_custom \
    --config_path config/efficientnet_b6_1280_resume.yaml \
    --in_base_dir input \
    --exp_name b6 \
    --save_checkpoint \
    --load_weights_only
```

This will initialize the new training run using the weights from the previous model while applying the settings defined in the new configuration file.

---

## 4. Output Directory

The output location for logs and checkpoints is defined in `train_custom.py` through the `out_dir` variable:

```python
out_dir = f"/content/drive/MyDrive/Visual Recognition/fp/{args.out_base_dir}/{args.exp_name}/{fold}"
```

### Current Output Path

```text
/content/drive/MyDrive/Visual Recognition/fp/{args.out_base_dir}/{args.exp_name}/{fold}
```

This directory is created directly in your Google Drive to ensure all the necessary data is not lost when the Google Colab session is terminated. All training logs, checkpoints, and related outputs will be saved to this directory.


In [ ]:
!python -m src.train_custom \
    --config_path config/efficientnet_b6.yaml \
    --in_base_dir input \
    --exp_name b6 \
    --save_checkpoint

# 8. Generate Predictions

## Prerequisites

Before generating predictions, please ensure the following:

### 1. A Checkpoint Is Available

Only proceed with this step if you already have a trained model checkpoint.

### 2. Checkpoint Filename

The checkpoint file **must be named exactly**:

```text
last.ckpt
```

This filename is required for the prediction script to locate the checkpoint correctly.

---

## Important Notes

### 3. Back Up Existing Files

Before generating predictions, make sure to back up any important output files.

Prediction outputs may overwrite existing files with the same name.

### 4. Verify Model and Configuration

Ensure that:

* The correct model is selected.
* The correct configuration file is specified.
* The checkpoint corresponds to the selected model configuration.

Using mismatched configurations may result in incorrect predictions.

---

## How Prediction Generation Works

### 5. No Training Is Performed

This step does **not** perform any training.

The prediction pipeline was specifically implemented by 歐力思 (111550202) to allow predictions to be generated directly from an existing model checkpoint without retraining the model.

This makes it possible to:

* Generate predictions from a previously trained model.
* Evaluate checkpoints without additional training.
* Quickly produce submission files or inference results.
* Save time and computational resources.


### 6. Results

After the cell execution is finished, you will be able to download .csv predictions in the ```/content/happywhale/kaggle-happywhale-1st-place/submissions``` folder. Submit it to the Kaggle sompetition (only .csv file, not pseudolabels)

In [ ]:
!python -m src.train_custom \
    --config_path config/efficientnet_b6.yaml \
    --in_base_dir input \
    --exp_name b6 \
    --load_snapshot \
    --skip_training

# 9. Ensemble Predictions

## Overview

This is the final step of the training and prediction pipeline.

You only need to perform this step if you would like to combine the predictions from multiple models to improve overall performance.

---

## Example Ensemble

For example, our best Kaggle submission was generated by ensembling the following models:

### EfficientNet-B6

* 1024 × 1024 resolution, 30 epochs
* 1280 × 1280 resolution, 15 epochs

### EfficientNet-B7

* 1280 × 1280 resolution, 30 epochs

The predictions from these models were combined to produce the final submission.

---

## Output Files

After the ensemble process is completed, the generated prediction files can be found in:

```text
/content/happywhale/kaggle-happywhale-1st-place/submissions
```

---

## Kaggle Submission

Download the generated submission file (`.csv`) from the submissions directory and upload it to the Kaggle competition.

### Important

Only submit the final prediction `.csv` file.

Do not submit any pseudolabel files, as they are intended for training purposes only and are not valid competition submissions.

---

## Pipeline Summary

The complete workflow is:

1. Train the model.
2. Generate predictions from the trained checkpoint.
3. Ensemble predictions from one or more models.
4. Download the final `.csv` submission file.
5. Submit the file to Kaggle.


In [ ]:
!python -m src.ensemble \
    --in_base_dir input \
    --model_dirs \
        /content/drive/MyDrive/Visual\ Recognition/fp/result/b7_1280_ft/-1 \
        /content/drive/MyDrive/Visual\ Recognition/fp/result/b6_1024_30_1280_15/-1 \
    --out_prefix b6_1024_30_1280_15_b7_1280_ft